In [1]:
import cv2
from pathlib import Path
import pandas as pd
import numpy as np
from skimage.draw import polygon
import matplotlib.pyplot as plt
import sys
from scipy import stats as scipy_stats


In [2]:
# now for ejection fraction
path_ef_ens = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/ensemble/results_ensemble_ef_adults.csv'
path_ef_phiseg = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/phiseg/results_phiseg_ef_adults.csv'
path_ef_vids = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/vids/results_vids_ef_adults.csv'

df_ef_ens = pd.read_csv(path_ef_ens)
df_ef_phiseg = pd.read_csv(path_ef_phiseg)
df_ef_vids = pd.read_csv(path_ef_vids)

In [3]:
bins = [-np.inf, 0, 2, 5, 12, 18]
labels = ['infant', 'toddler', 'preschooler', 'school age', 'teenager']

df_ef_ens['age_group'] = pd.cut(df_ef_ens['age'], bins=bins, labels=labels, right=True)

# Absolute error
df_ef_ens['abs_error'] = np.abs(df_ef_ens['mean_ef_pred'] - df_ef_ens['gt_ef'])

# Uncertainty interval: [mean_ef_pred - std_ef_pred, mean_ef_pred + std_ef_pred]
# Coverage: does gt_ef fall inside this interval?
df_ef_ens['covered'] = (
    (df_ef_ens['gt_ef'] >= df_ef_ens['mean_ef_pred'] - df_ef_ens['std_ef_pred']) &
    (df_ef_ens['gt_ef'] <= df_ef_ens['mean_ef_pred'] + df_ef_ens['std_ef_pred'])
)

# Interval size = 2 * std_ef_pred
df_ef_ens['interval_size'] = 2 * df_ef_ens['std_ef_pred']

# Compute stats per age group
def compute_uncertainty_stats(group):
    result = {}

    # Coverage
    result['coverage'] = group['covered'].mean()

    # Interval size stats
    result['mean_interval_size'] = group['interval_size'].mean()
    result['std_interval_size'] = group['interval_size'].std()

    # Correlation between interval size and absolute error
    result['pearson_r'] = group['interval_size'].corr(group['abs_error'])
    if len(group) > 2:
        spearman_r, spearman_p = scipy_stats.spearmanr(group['interval_size'], group['abs_error'])
        result['spearman_r'] = spearman_r
        result['spearman_p'] = spearman_p
    else:
        result['spearman_r'] = np.nan
        result['spearman_p'] = np.nan

    result['n_samples'] = len(group)
    return pd.Series(result)

stats = df_ef_ens.groupby('age_group', observed=False).apply(compute_uncertainty_stats)
print(stats.round(3))

             coverage  mean_interval_size  std_interval_size  pearson_r  \
age_group                                                                 
infant          0.500              31.206             52.806      0.939   
toddler         0.483              17.517             12.984      0.290   
preschooler     0.500              12.044             10.978      0.579   
school age      0.350              16.525             39.106      0.933   
teenager        0.237              14.751             19.880      0.804   

             spearman_r  spearman_p  n_samples  
age_group                                       
infant            0.552       0.063       12.0  
toddler           0.451       0.014       29.0  
preschooler       0.241       0.125       42.0  
school age        0.382       0.000      100.0  
teenager          0.367       0.000      131.0  


/tmp/ipykernel_318283/1895698631.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats = df_ef_ens.groupby('age_group', observed=False).apply(compute_uncertainty_stats)


In [4]:
bins = [-np.inf, 0, 2, 5, 12, 18]
labels = ['infant', 'toddler', 'preschooler', 'school age', 'teenager']

df_ef_phiseg['age_group'] = pd.cut(df_ef_phiseg['age'], bins=bins, labels=labels, right=True)

# Absolute error
df_ef_phiseg['abs_error'] = np.abs(df_ef_phiseg['mean_ef_pred'] - df_ef_phiseg['gt_ef'])

# Uncertainty interval: [mean_ef_pred - std_ef_pred, mean_ef_pred + std_ef_pred]
# Coverage: does gt_ef fall inside this interval?
df_ef_phiseg['covered'] = (
    (df_ef_phiseg['gt_ef'] >= df_ef_phiseg['mean_ef_pred'] - df_ef_phiseg['std_ef_pred']) &
    (df_ef_phiseg['gt_ef'] <= df_ef_phiseg['mean_ef_pred'] + df_ef_phiseg['std_ef_pred'])
)

# Interval size = 2 * std_ef_pred
df_ef_phiseg['interval_size'] = 2 * df_ef_phiseg['std_ef_pred']

# Compute stats per age group
def compute_uncertainty_stats(group):
    result = {}

    # Coverage
    result['coverage'] = group['covered'].mean()

    # Interval size stats
    result['mean_interval_size'] = group['interval_size'].mean()
    result['std_interval_size'] = group['interval_size'].std()

    # Correlation between interval size and absolute error
    result['pearson_r'] = group['interval_size'].corr(group['abs_error'])
    if len(group) > 2:
        spearman_r, spearman_p = scipy_stats.spearmanr(group['interval_size'], group['abs_error'])
        result['spearman_r'] = spearman_r
        result['spearman_p'] = spearman_p
    else:
        result['spearman_r'] = np.nan
        result['spearman_p'] = np.nan

    result['n_samples'] = len(group)
    return pd.Series(result)

stats = df_ef_phiseg.groupby('age_group', observed=False).apply(compute_uncertainty_stats)
print(stats.round(3))

             coverage  mean_interval_size  std_interval_size  pearson_r  \
age_group                                                                 
infant          0.583              37.479             40.032      0.562   
toddler         0.483              25.310             17.762      0.661   
preschooler     0.571              21.761             15.538      0.793   
school age      0.570              24.671             42.235      0.686   
teenager        0.550              32.257             92.704      0.927   

             spearman_r  spearman_p  n_samples  
age_group                                       
infant            0.643       0.024       12.0  
toddler           0.248       0.195       29.0  
preschooler       0.262       0.093       42.0  
school age        0.475       0.000      100.0  
teenager          0.479       0.000      131.0  


/tmp/ipykernel_318283/2515250232.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats = df_ef_phiseg.groupby('age_group', observed=False).apply(compute_uncertainty_stats)


In [5]:
bins = [-np.inf, 0, 2, 5, 12, 18]
labels = ['infant', 'toddler', 'preschooler', 'school age', 'teenager']

df_ef_vids['age_group'] = pd.cut(df_ef_vids['age'], bins=bins, labels=labels, right=True)

# Absolute error
df_ef_vids['abs_error'] = np.abs(df_ef_vids['mean_ef_pred'] - df_ef_vids['gt_ef'])

# Uncertainty interval: [mean_ef_pred - std_ef_pred, mean_ef_pred + std_ef_pred]
# Coverage: does gt_ef fall inside this interval?
df_ef_vids['covered'] = (
    (df_ef_vids['gt_ef'] >= df_ef_vids['mean_ef_pred'] - df_ef_vids['std_ef_pred']) &
    (df_ef_vids['gt_ef'] <= df_ef_vids['mean_ef_pred'] + df_ef_vids['std_ef_pred'])
)

# Interval size = 2 * std_ef_pred
df_ef_vids['interval_size'] = 2 * df_ef_vids['std_ef_pred']

# Compute stats per age group
def compute_uncertainty_stats(group):
    result = {}

    # Coverage
    result['coverage'] = group['covered'].mean()

    # Interval size stats
    result['mean_interval_size'] = group['interval_size'].mean()
    result['std_interval_size'] = group['interval_size'].std()

    # Correlation between interval size and absolute error
    result['pearson_r'] = group['interval_size'].corr(group['abs_error'])
    if len(group) > 2:
        spearman_r, spearman_p = scipy_stats.spearmanr(group['interval_size'], group['abs_error'])
        result['spearman_r'] = spearman_r
        result['spearman_p'] = spearman_p
    else:
        result['spearman_r'] = np.nan
        result['spearman_p'] = np.nan

    result['n_samples'] = len(group)
    return pd.Series(result)

stats = df_ef_vids.groupby('age_group', observed=False).apply(compute_uncertainty_stats)
print(stats.round(3))

             coverage  mean_interval_size  std_interval_size  pearson_r  \
age_group                                                                 
infant          0.000               0.970              2.093      0.948   
toddler         0.000               0.671              1.174      0.707   
preschooler     0.024               0.298              0.496     -0.001   
school age      0.000               0.484              2.337      0.498   
teenager        0.015               0.416              0.751      0.206   

             spearman_r  spearman_p  n_samples  
age_group                                       
infant            0.343       0.276       12.0  
toddler           0.467       0.011       29.0  
preschooler       0.070       0.661       42.0  
school age        0.303       0.002      100.0  
teenager          0.195       0.025      131.0  


/tmp/ipykernel_318283/1417930724.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats = df_ef_vids.groupby('age_group', observed=False).apply(compute_uncertainty_stats)
